In [41]:
%store -r

In [82]:
import boto3
import sagemaker
# Import CatClsDataset class from local python file

from cat_dataset import CatClsDataset


sess = sagemaker.Session()
bucket = sess.default_bucket()
role = sagemaker.get_execution_role()
region = boto3.Session().region_name

s3 = boto3.client("s3")

In [43]:

database_name = "cat_image_analysis"
table_name = "image_landmarks_features"

region = "us-east-1"

s3_combined_location = (
    f"s3://{bucket}/cat-landmarks-project/processed/combined/image_landmarks_features/"
)

s3_staging_dir = f"s3://{bucket}/athena/staging/"


In [44]:
from pyathena import connect
conn = connect(region_name=region, s3_staging_dir=s3_staging_dir)

In [45]:
from pyathena import connect

conn = connect(
    region_name=region,
    s3_staging_dir=s3_staging_dir
)

cursor = conn.cursor()



In [46]:
import pandas as pd
statement = f"""
SELECT *
FROM {database_name}.{table_name} 
"""
print(statement)


df_features = pd.read_sql(statement, conn)
df_features.head(2)



SELECT *
FROM cat_image_analysis.image_landmarks_features 



/tmp/ipykernel_2036/579769470.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_features = pd.read_sql(statement, conn)


,image_id,label,width,height,aspect_ratio,area,eye_center_x_norm,eye_center_y_norm,eye_dist_norm,eye_y_diff_norm,eye_angle,mouth_x_norm,mouth_y_norm,mouth_eye_y_norm,dataset_split,event_time
0,s3://sagemaker-us-east-1-549206572067/cat-land...,1,375,500,0.750000,187500,0.552,0.322000,0.170667,0.004000,0.031240,0.530667,0.398000,0.076000,train,1771005883.05649
1,s3://sagemaker-us-east-1-549206572067/cat-land...,1,500,333,1.501502,166500,0.294,0.364865,0.128000,0.003003,0.292562,0.266000,0.507508,0.142643,train,1771005883.05649


## Feature Store

In [47]:
import boto3
import sagemaker

original_boto3_version = boto3.__version__
%pip install 'boto3>1.17.21'

Note: you may need to restart the kernel to use updated packages.


In [48]:
from sagemaker.session import Session

region = boto3.Session().region_name

boto_session = boto3.Session(region_name=region)

sagemaker_client = boto_session.client(service_name="sagemaker", region_name=region)
featurestore_runtime = boto_session.client(
    service_name="sagemaker-featurestore-runtime", region_name=region
)

feature_store_session = Session(
    boto_session=boto_session,
    sagemaker_client=sagemaker_client,
    sagemaker_featurestore_runtime_client=featurestore_runtime,
)

In [49]:

database_name = "cat_image_analysis"
table_name = "image_landmarks_features"
region = "us-east-1"
print(s3_bucket)
print (project_prefix )


sagemaker-us-east-1-549206572067
cat-landmarks-project


In [50]:
from sagemaker import get_execution_role

role = get_execution_role()
print(role)

arn:aws:iam::549206572067:role/service-role/AmazonSageMaker-ExecutionRole-20260128T205128


In [51]:
# -----------------------------------------
# Load engineered features from Athena table
# -----------------------------------------
import pandas as pd

query = f"""
SELECT
image_id,
label,
width,
height,
aspect_ratio,
area,
eye_center_x_norm,
eye_center_y_norm,
eye_dist_norm,
eye_y_diff_norm,
eye_angle,
mouth_x_norm,
mouth_y_norm,
mouth_eye_y_norm,
dataset_split,
event_time
FROM {database_name}.{table_name}
"""

df_catlm_fs = pd.read_sql(query, conn)

df_catlm_fs["event_time"] = df_catlm_fs["event_time"].astype("float64")

df_catlm_fs.head(2)


/tmp/ipykernel_2036/245513661.py:27: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_catlm_fs = pd.read_sql(query, conn)


,image_id,label,width,height,aspect_ratio,area,eye_center_x_norm,eye_center_y_norm,eye_dist_norm,eye_y_diff_norm,eye_angle,mouth_x_norm,mouth_y_norm,mouth_eye_y_norm,dataset_split,event_time
0,s3://sagemaker-us-east-1-549206572067/cat-land...,1,375,500,0.750000,187500,0.552,0.322000,0.170667,0.004000,0.031240,0.530667,0.398000,0.076000,train,1.771006e+09
1,s3://sagemaker-us-east-1-549206572067/cat-land...,1,500,333,1.501502,166500,0.294,0.364865,0.128000,0.003003,0.292562,0.266000,0.507508,0.142643,train,1.771006e+09


## Ingest Data into FeatureStore

In [52]:
df_catlm_fs.count()

image_id             33109
label                33109
width                33109
height               33109
aspect_ratio         33109
area                 33109
eye_center_x_norm    18109
eye_center_y_norm    18109
eye_dist_norm        18109
eye_y_diff_norm      18109
eye_angle             4140
mouth_x_norm         18109
mouth_y_norm         18109
mouth_eye_y_norm     18109
dataset_split        33109
event_time           33109
dtype: int64

In [53]:
from time import gmtime, strftime, sleep

combined_landmark_feature_group_name = "combined-landmark-feature-group-" + strftime("%d-%H-%M-%S", gmtime())

In [54]:
from sagemaker.feature_store.feature_group import FeatureGroup

combined_landmark_feature_group = FeatureGroup(
    name=combined_landmark_feature_group_name, sagemaker_session=feature_store_session
)

In [55]:
import time

current_time_sec = int(round(time.time()))


def cast_object_to_string(data_frame):
    for label in data_frame.columns:
        if data_frame.dtypes[label] == "object":
            data_frame[label] = data_frame[label].astype("str").astype("string")


record_identifier_feature_name = "image_id"
event_time_feature_name = "event_time"

# sanity checks 
if record_identifier_feature_name not in df_catlm_fs.columns:
    raise ValueError("image_id is missing. Create it before Feature Store ingestion.")

if df_catlm_fs[record_identifier_feature_name].isna().any():
    raise ValueError("image_id contains nulls. Feature Store record identifier cannot be null.")

if not df_catlm_fs[record_identifier_feature_name].is_unique:
    raise ValueError("image_id must be unique. You have duplicates (would overwrite records).")

# cast object dtype 
cast_object_to_string(df_catlm_fs)

# ---- load feature definitions (schema inference)
combined_landmark_feature_group.load_feature_definitions(data_frame=df_catlm_fs)




[FeatureDefinition(feature_name='image_id', feature_type=<FeatureTypeEnum.STRING: 'String'>, collection_type=None),
 FeatureDefinition(feature_name='label', feature_type=<FeatureTypeEnum.INTEGRAL: 'Integral'>, collection_type=None),
 FeatureDefinition(feature_name='width', feature_type=<FeatureTypeEnum.INTEGRAL: 'Integral'>, collection_type=None),
 FeatureDefinition(feature_name='height', feature_type=<FeatureTypeEnum.INTEGRAL: 'Integral'>, collection_type=None),
 FeatureDefinition(feature_name='aspect_ratio', feature_type=<FeatureTypeEnum.FRACTIONAL: 'Fractional'>, collection_type=None),
 FeatureDefinition(feature_name='area', feature_type=<FeatureTypeEnum.INTEGRAL: 'Integral'>, collection_type=None),
 FeatureDefinition(feature_name='eye_center_x_norm', feature_type=<FeatureTypeEnum.FRACTIONAL: 'Fractional'>, collection_type=None),
 FeatureDefinition(feature_name='eye_center_y_norm', feature_type=<FeatureTypeEnum.FRACTIONAL: 'Fractional'>, collection_type=None),
 FeatureDefinition(fea

In [56]:
def wait_for_feature_group_creation_complete(feature_group):
    status = feature_group.describe().get("FeatureGroupStatus")
    while status == "Creating":
        print("Waiting for Feature Group Creation")
        time.sleep(5)
        status = feature_group.describe().get("FeatureGroupStatus")
    if status != "Created":
        raise RuntimeError(f"Failed to create feature group {feature_group.name}")
    print(f"FeatureGroup {feature_group.name} successfully created.")


combined_landmark_feature_group.create(
    s3_uri=f"s3://{s3_bucket}/{project_prefix}",
    record_identifier_name=record_identifier_feature_name,
    event_time_feature_name=event_time_feature_name,
    role_arn=role,
    enable_online_store=True,
)


wait_for_feature_group_creation_complete(feature_group=combined_landmark_feature_group)

Waiting for Feature Group Creation
Waiting for Feature Group Creation
Waiting for Feature Group Creation
Waiting for Feature Group Creation
Waiting for Feature Group Creation
FeatureGroup combined-landmark-feature-group-13-20-14-41 successfully created.


In [57]:
combined_landmark_feature_group.describe()

{'FeatureGroupArn': 'arn:aws:sagemaker:us-east-1:549206572067:feature-group/combined-landmark-feature-group-13-20-14-41',
 'FeatureGroupName': 'combined-landmark-feature-group-13-20-14-41',
 'RecordIdentifierFeatureName': 'image_id',
 'EventTimeFeatureName': 'event_time',
 'FeatureDefinitions': [{'FeatureName': 'image_id', 'FeatureType': 'String'},
  {'FeatureName': 'label', 'FeatureType': 'Integral'},
  {'FeatureName': 'width', 'FeatureType': 'Integral'},
  {'FeatureName': 'height', 'FeatureType': 'Integral'},
  {'FeatureName': 'aspect_ratio', 'FeatureType': 'Fractional'},
  {'FeatureName': 'area', 'FeatureType': 'Integral'},
  {'FeatureName': 'eye_center_x_norm', 'FeatureType': 'Fractional'},
  {'FeatureName': 'eye_center_y_norm', 'FeatureType': 'Fractional'},
  {'FeatureName': 'eye_dist_norm', 'FeatureType': 'Fractional'},
  {'FeatureName': 'eye_y_diff_norm', 'FeatureType': 'Fractional'},
  {'FeatureName': 'eye_angle', 'FeatureType': 'Fractional'},
  {'FeatureName': 'mouth_x_norm', 

In [58]:
combined_landmark_feature_group.ingest(data_frame=df_catlm_fs, max_workers=3, wait=True)

IngestionManagerPandas(feature_group_name='combined-landmark-feature-group-13-20-14-41', feature_definitions={'image_id': {'FeatureName': 'image_id', 'FeatureType': 'String'}, 'label': {'FeatureName': 'label', 'FeatureType': 'Integral'}, 'width': {'FeatureName': 'width', 'FeatureType': 'Integral'}, 'height': {'FeatureName': 'height', 'FeatureType': 'Integral'}, 'aspect_ratio': {'FeatureName': 'aspect_ratio', 'FeatureType': 'Fractional'}, 'area': {'FeatureName': 'area', 'FeatureType': 'Integral'}, 'eye_center_x_norm': {'FeatureName': 'eye_center_x_norm', 'FeatureType': 'Fractional'}, 'eye_center_y_norm': {'FeatureName': 'eye_center_y_norm', 'FeatureType': 'Fractional'}, 'eye_dist_norm': {'FeatureName': 'eye_dist_norm', 'FeatureType': 'Fractional'}, 'eye_y_diff_norm': {'FeatureName': 'eye_y_diff_norm', 'FeatureType': 'Fractional'}, 'eye_angle': {'FeatureName': 'eye_angle', 'FeatureType': 'Fractional'}, 'mouth_x_norm': {'FeatureName': 'mouth_x_norm', 'FeatureType': 'Fractional'}, 'mouth_y

In [59]:
#  Verifying with feature record

record_identifier_value = "s3://sagemaker-us-east-1-549206572067/cat-landmarks-project/raw/cats/images/CAT_00/00000001_000.jpg"

fg_name = combined_landmark_feature_group.name  

response = featurestore_runtime.get_record(
    FeatureGroupName=fg_name,
    RecordIdentifierValueAsString=record_identifier_value,
)

print(response["Record"])



[{'FeatureName': 'image_id', 'ValueAsString': 's3://sagemaker-us-east-1-549206572067/cat-landmarks-project/raw/cats/images/CAT_00/00000001_000.jpg'}, {'FeatureName': 'label', 'ValueAsString': '1'}, {'FeatureName': 'width', 'ValueAsString': '375'}, {'FeatureName': 'height', 'ValueAsString': '500'}, {'FeatureName': 'aspect_ratio', 'ValueAsString': '0.75'}, {'FeatureName': 'area', 'ValueAsString': '187500'}, {'FeatureName': 'eye_center_x_norm', 'ValueAsString': '0.552'}, {'FeatureName': 'eye_center_y_norm', 'ValueAsString': '0.322'}, {'FeatureName': 'eye_dist_norm', 'ValueAsString': '0.17066666666666666'}, {'FeatureName': 'eye_y_diff_norm', 'ValueAsString': '0.0040000000000000036'}, {'FeatureName': 'eye_angle', 'ValueAsString': '0.031239833430268277'}, {'FeatureName': 'mouth_x_norm', 'ValueAsString': '0.5306666666666666'}, {'FeatureName': 'mouth_y_norm', 'ValueAsString': '0.398'}, {'FeatureName': 'mouth_eye_y_norm', 'ValueAsString': '0.07600000000000001'}, {'FeatureName': 'dataset_split',

In [60]:

import sagemaker
import boto3

sagemaker_session = sagemaker.Session()
region = boto3.Session().region_name

# Get execution role (works inside SageMaker notebooks)
role = sagemaker.get_execution_role()

print("Region:", region)
print("Role:", role)


Region: us-east-1
Role: arn:aws:iam::549206572067:role/service-role/AmazonSageMaker-ExecutionRole-20260128T205128


In [61]:
from sagemaker.feature_store.feature_group import FeatureGroup

combined_landmark_feature_group = FeatureGroup(
    name="combined_landmark_feature_group",
    sagemaker_session=sagemaker_session
)


In [62]:

from sagemaker.feature_store.feature_group import FeatureGroup
import sagemaker

sagemaker_session = sagemaker.Session()

combined_landmark_feature_group = FeatureGroup(
    name= combined_landmark_feature_group_name,
    sagemaker_session=sagemaker_session
)

desc = combined_landmark_feature_group.describe()
print(desc["FeatureGroupStatus"])



Created


In [63]:
# Lists existing Feature Groups in your account/region

import boto3

sm = boto3.client("sagemaker")

resp = sm.list_feature_groups(MaxResults=50)
for fg in resp["FeatureGroupSummaries"]:
    print(fg["FeatureGroupName"])
    

combined-landmark-feature-group-13-20-14-41
combined-landmark-feature-group-13-20-03-53
combined-landmark-feature-group-13-18-08-27
combined-landmark-feature-group-13-06-01-36
combined-landmark-feature-group-13-02-36-52
combined-landmark-feature-group-12-04-01-49
combined-landmark-feature-group-12-02-38-30
combined-landmark-feature-group-01-23-17-25
combined-feature-group-01-23-09-39
combined-feature-group-01-22-33-03
combined-feature-group-01-07-36-12
combined-feature-group-01-05-14-57
catlm-feature-group-01-05-06-17
catlm-feature-group-01-04-44-17


In [64]:
offline_cfg = desc.get("OfflineStoreConfig", {})
print(offline_cfg)


{'S3StorageConfig': {'S3Uri': 's3://sagemaker-us-east-1-549206572067/cat-landmarks-project', 'ResolvedOutputS3Uri': 's3://sagemaker-us-east-1-549206572067/cat-landmarks-project/549206572067/sagemaker/us-east-1/offline-store/combined-landmark-feature-group-13-20-14-41-1771013688/data'}, 'DisableGlueTableCreation': False, 'DataCatalogConfig': {'TableName': 'combined_landmark_feature_group_13_20_14_41_1771013688', 'Catalog': 'AwsDataCatalog', 'Database': 'sagemaker_featurestore'}}


### Builing Datasets for Modelling

In [70]:
# Robust offline-store fetch by split.
# Pull split-specific datasets from the offline store (Athena)
# Feature Store Offline Store pipeline (Athena)




ATHENA_OUTPUT = "s3://sagemaker-us-east-1-549206572067/athena-results/"
prefix = "cat-landmarks-project/feature-store"
combined_query = combined_landmark_feature_group.athena_query()


ATHENA_OUTPUT = "s3://sagemaker-us-east-1-549206572067/athena-results/"

combined_query = combined_landmark_feature_group.athena_query()
def fetch_split(split_name: str):
    q = f"""
    SELECT *
    FROM "{combined_query.database}"."{combined_query.table_name}"
    WHERE lower(trim(dataset_split)) = '{split_name.lower()}'
    """

    combined_query.run(query_string=q, output_location=ATHENA_OUTPUT)
    combined_query.wait()

    df = combined_query.as_dataframe()
    print(f"{split_name} rows:", len(df))
    return df

train_df = fetch_split("train")
val_df   = fetch_split("val")
test_df  = fetch_split("test")
prod_df  = fetch_split("prod")



# Renaming after creation
rename_map = {"image_id": "s3_uri"}
train_df = train_df.rename(columns=rename_map)
val_df   = val_df.rename(columns=rename_map)
test_df  = test_df.rename(columns=rename_map)
prod_df  = prod_df.rename(columns=rename_map)

val_df.head(1)

train rows: 18025
val rows: 2525
test rows: 2544
prod rows: 10015


,s3_uri,label,width,height,aspect_ratio,area,eye_center_x_norm,eye_center_y_norm,eye_dist_norm,eye_y_diff_norm,eye_angle,mouth_x_norm,mouth_y_norm,mouth_eye_y_norm,dataset_split,event_time,write_time,api_invocation_time,is_deleted
0,s3://sagemaker-us-east-1-549206572067/cat-land...,1,499,281,1.775801,140219,0.214429,0.455516,0.084168,0.014235,NaN,0.192385,0.590747,0.135231,val,1.771006e+09,2026-02-13 20:21:15.922,2026-02-13 20:17:21.000,False


In [71]:
#  Create datasets from Athena-fetched DataFrames

train_ds = CatClsDataset(train_df, augment=False, aug=None, image_size=(224,224))
val_ds   = CatClsDataset(val_df,   augment=False, aug=None, image_size=(224,224))
test_ds  = CatClsDataset(test_df,  augment=False, aug=None, image_size=(224,224))
prod_ds  = CatClsDataset(prod_df,  augment=False, aug=None, image_size=(224,224))

In [72]:
# DataLoaders feed mini-batches to the training loop


from torch.utils.data import DataLoader

BATCH_SIZE = 32

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
prod_loader  = DataLoader(prod_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)


## Benchmark

In [54]:
# Simple heuristic benchmark using one feature for Landmark model comparision

def heuristic_predict(df):
    # predict cat if eye distance is not NaN and above a small threshold
    return ((df["eye_dist_norm"].notna()) & (df["eye_dist_norm"] > 0.05)).astype(int)


In [55]:
# Evaluate heuristic benchmark

from sklearn.metrics import accuracy_score, classification_report

bench_pred = heuristic_predict(test_df)
bench_acc = accuracy_score(test_df["label"].astype(int), bench_pred)

print("Heuristic benchmark test acc:", bench_acc)
print(classification_report(test_df["label"].astype(int), bench_pred))


Heuristic benchmark test acc: 0.9732704402515723
              precision    recall  f1-score   support

           0       0.96      1.00      0.98      3024
           1       1.00      0.93      0.97      2064

    accuracy                           0.97      5088
   macro avg       0.98      0.97      0.97      5088
weighted avg       0.97      0.97      0.97      5088



In [75]:
# Simple image-based baseline using mean brightness
# Uses stratified sampling with safe sizes
# Reads images but stores only ONE float per image (low RAM)
# Tunes threshold on train subset, evaluates on test subset

import numpy as np
from PIL import Image
import boto3, io
from sklearn.metrics import accuracy_score, classification_report

s3 = boto3.client("s3")

TARGET_TRAIN = 10000
TARGET_TEST  = 3000

def stratified_subset(df, target_n, label_col="label", seed=42):
    # Take up to target_n rows, preserving label proportions
    n = min(target_n, len(df))
    counts = df[label_col].value_counts()
    # Allocate per-class sample counts proportional to class distribution
    per_class = (counts / counts.sum() * n).round().astype(int)
    # Fix rounding drift so total equals n
    drift = n - per_class.sum()
    if drift != 0:
        # adjust the largest class
        per_class.iloc[0] += drift

    parts = []
    for label, k in per_class.items():
        k = min(int(k), (df[label_col] == label).sum())
        parts.append(df[df[label_col] == label].sample(n=k, random_state=seed))
    return np.random.RandomState(seed).permutation(
        np.concatenate([p.index.values for p in parts])
    )

train_idx = stratified_subset(train_df, TARGET_TRAIN)
test_idx  = stratified_subset(test_df,  TARGET_TEST)

train_subset = train_df.loc[train_idx].reset_index(drop=True)
test_subset  = test_df.loc[test_idx].reset_index(drop=True)

print("Train subset:", len(train_subset), "Label counts:", train_subset["label"].value_counts().to_dict())
print("Test subset:", len(test_subset), "Label counts:", test_subset["label"].value_counts().to_dict())

def mean_brightness_from_s3(s3_uri: str, size=(64, 64)) -> float:
    bucket, key = s3_uri.replace("s3://", "").split("/", 1)
    obj = s3.get_object(Bucket=bucket, Key=key)
    img = Image.open(io.BytesIO(obj["Body"].read())).convert("L").resize(size)
    arr = np.asarray(img, dtype=np.float32)
    return float(arr.mean())

x_train = np.array([mean_brightness_from_s3(u) for u in train_subset["s3_uri"].tolist()], dtype=np.float32)
y_train = train_subset["label"].astype(int).to_numpy()

x_test  = np.array([mean_brightness_from_s3(u) for u in test_subset["s3_uri"].tolist()], dtype=np.float32)
y_test  = test_subset["label"].astype(int).to_numpy()

# Tune threshold on train
candidates = np.unique(x_train)
best_acc, best_t = -1.0, None
step = max(1, len(candidates)//200)

for t in candidates[::step]:
    pred = (x_train >= t).astype(int)
    acc = accuracy_score(y_train, pred)
    if acc > best_acc:
        best_acc, best_t = acc, float(t)

# Evaluate
pred_test = (x_test >= best_t).astype(int)

print("Brightness-threshold baseline | best_t:", best_t, "| train_acc:", best_acc)
print("Test acc:", accuracy_score(y_test, pred_test))
print(classification_report(y_test, pred_test, zero_division=0))


Train subset: 10000 Label counts: {1: 6751, 0: 3249}
Test subset: 2544 Label counts: {0: 1512, 1: 1032}
Brightness-threshold baseline | best_t: 6.69287109375 | train_acc: 0.6751
Test acc: 0.4056603773584906
              precision    recall  f1-score   support

           0       0.00      0.00      0.00      1512
           1       0.41      1.00      0.58      1032

    accuracy                           0.41      2544
   macro avg       0.20      0.50      0.29      2544
weighted avg       0.16      0.41      0.23      2544



In [76]:
#  Save brightness baseline metrics to CSV
# Useful for experiment tracking and reporting

import pandas as pd
from datetime import datetime
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

acc = accuracy_score(y_test, pred_test)
p, r, f1, _ = precision_recall_fscore_support(
    y_test, pred_test, average="weighted", zero_division=0
)

benchmark_row = {
    "timestamp": datetime.utcnow().isoformat(),
    "model_name": "brightness_threshold_baseline",
    "split": "test",
    "accuracy": float(acc),
    "precision_weighted": float(p),
    "recall_weighted": float(r),
    "f1_weighted": float(f1),
    "train_subset_size": len(train_subset),
    "test_subset_size": len(test_subset),
    "best_threshold": float(best_t),
    "notes": "64x64 grayscale mean brightness baseline"
}

results_path = "/tmp/benchmark_results.csv"

try:
    existing = pd.read_csv(results_path)
    out = pd.concat([existing, pd.DataFrame([benchmark_row])], ignore_index=True)
except FileNotFoundError:
    out = pd.DataFrame([benchmark_row])

out.to_csv(results_path, index=False)
print("Saved benchmark to:", results_path)


Saved benchmark to: /tmp/benchmark_results.csv


/tmp/ipykernel_2036/1251542699.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat(),


In [96]:
#  Upload saved benchmark CSV to S3 for persistence

import boto3

s3 = boto3.client("s3")
bucket = sess.default_bucket()

key = "cat-landmarks-project/benchmarks/brightness_baseline_v1.csv"
s3.upload_file(results_path, bucket, key)

print(f"Uploaded to s3://{bucket}/{key}")


Uploaded to s3://sagemaker-us-east-1-549206572067/cat-landmarks-project/benchmarks/brightness_baseline_v1.csv


### Modelling for Classification

In [97]:
# Model SmallCNN
# Define a small CNN and optimizer for binary classification

import torch
import torch.nn as nn
import torch.nn.functional as F

class SmallCNN(nn.Module):
    def __init__(self, num_classes: int = 2):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = self.gap(x).squeeze(-1).squeeze(-1)
        return self.fc(x)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = SmallCNN(num_classes=2).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print("Device:", device)


Device: cpu


In [98]:
# Simple Dataset for classification only (no landmark columns needed)
# Expects Feature Store schema: image_id contains s3:// URI, label is 0/1

import io
import boto3
import numpy as np
from PIL import Image
import torch
from torch.utils.data import Dataset

_s3 = boto3.client("s3")

def read_image_from_s3_uri(s3_uri: str) -> Image.Image:
    bucket, key = s3_uri.replace("s3://", "").split("/", 1)
    obj = _s3.get_object(Bucket=bucket, Key=key)
    return Image.open(io.BytesIO(obj["Body"].read())).convert("RGB")

class CatClsDatasetCls(Dataset):
    def __init__(self, df, image_size=(224,224), uri_col="image_id", label_col="label"):
        self.df = df.reset_index(drop=True)
        self.image_size = tuple(image_size)
        self.uri_col = uri_col
        self.label_col = label_col

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        s3_uri = row[self.uri_col]
        label = int(row[self.label_col])

        img = read_image_from_s3_uri(s3_uri)
        img = img.resize(self.image_size)

        arr = np.array(img).astype(np.float32) / 255.0  # HWC
        x = torch.from_numpy(arr).permute(2, 0, 1)      # CHW
        y = torch.tensor(label, dtype=torch.long)
        return x, y


In [99]:
# Use classification-only dataset for CNN training
# Use the correct URI column name when building the Dataset

train_ds_cls = CatClsDatasetCls(train_df, image_size=(224,224), uri_col="s3_uri", label_col="label")
val_ds_cls   = CatClsDatasetCls(val_df,   image_size=(224,224), uri_col="s3_uri", label_col="label")
test_ds_cls  = CatClsDatasetCls(test_df,  image_size=(224,224), uri_col="s3_uri", label_col="label")
prod_ds_cls  = CatClsDatasetCls(prod_df,  image_size=(224,224), uri_col="s3_uri", label_col="label")


In [100]:
#  DataLoaders create mini-batches for training / validation / testing

from torch.utils.data import DataLoader

BATCH_SIZE = 32

train_loader_cls = DataLoader(train_ds_cls, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader_cls   = DataLoader(val_ds_cls,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader_cls  = DataLoader(test_ds_cls,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
prod_loader_cls  = DataLoader(prod_ds,      batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)




### Model Training

In [101]:
# Training loop 
import io
import boto3
from PIL import Image


    
def run_one_epoch(model, loader, train_mode: bool):
    model.train() if train_mode else model.eval()

    total_loss, correct, total = 0.0, 0, 0

    with torch.set_grad_enabled(train_mode):
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)

            logits = model(xb)
            loss = criterion(logits, yb)

            if train_mode:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += float(loss.item()) * xb.size(0)
            preds = torch.argmax(logits, dim=1)
            correct += int((preds == yb).sum().item())
            total += xb.size(0)

    return total_loss / max(total, 1), correct / max(total, 1)
metrics = []
EPOCHS = 4
for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_one_epoch(model, train_loader_cls, train_mode=True)
    va_loss, va_acc = run_one_epoch(model, val_loader_cls,   train_mode=False)
    print(f"epoch={epoch} train_loss={tr_loss:.4f} train_acc={tr_acc:.4f} val_loss={va_loss:.4f} val_acc={va_acc:.4f}")
    metrics.append({
        "epoch": epoch,
        "train_loss": tr_loss,
        "train_acc": tr_acc,
        "val_loss": va_loss,
        "val_acc": va_acc
    })


epoch=1 train_loss=0.5903 train_acc=0.6971 val_loss=0.7685 val_acc=0.5350
epoch=2 train_loss=0.5047 train_acc=0.7618 val_loss=0.7177 val_acc=0.5917
epoch=3 train_loss=0.4716 train_acc=0.7829 val_loss=0.7662 val_acc=0.5937
epoch=4 train_loss=0.4531 train_acc=0.7958 val_loss=0.6317 val_acc=0.7006


In [102]:
# Save CNN training metrics locally

metrics_df = pd.DataFrame(metrics)

local_path = "/tmp/cnn_training_metrics_v1.csv"
metrics_df.to_csv(local_path, index=False)

print("Saved locally:", local_path)
metrics_df.tail()


Saved locally: /tmp/cnn_training_metrics_v1.csv


,epoch,train_loss,train_acc,val_loss,val_acc
0,1,0.590286,0.697143,0.768487,0.535050
1,2,0.504689,0.761775,0.717741,0.591683
2,3,0.471621,0.782857,0.766204,0.593663
3,4,0.453121,0.795784,0.631670,0.700594


In [103]:
#  Upload CNN metrics to S3 for persistence

import boto3

s3 = boto3.client("s3")

key = "cat-landmarks-project/benchmarks/cnn_training_metrics_v1.csv"

s3.upload_file(local_path, s3_bucket, key)

print(f"Uploaded to s3://{s3_bucket}/{key}")


Uploaded to s3://sagemaker-us-east-1-549206572067/cat-landmarks-project/benchmarks/cnn_training_metrics_v1.csv


In [ ]:
# Comments:
# - Use instance-safe temp directory
# - Avoid hardcoding home path

import os

model_dir = "/tmp"
model_path = os.path.join(model_dir, "model_cls.pth")

torch.save(model.state_dict(), model_path)

print("Saved trained model to:", model_path)


In [105]:
# Evaluate on test set once after training

te_loss, te_acc = run_one_epoch(model, test_loader_cls, train_mode=False)
print("Test loss:", te_loss, "Test acc:", te_acc)
metrics.append({
    "epoch": "test_final",
    "test_loss": te_loss,
    "test_acc": te_acc,
})

metrics_df = pd.DataFrame(metrics)
metrics_df.to_csv(local_path, index=False)

# Now upload the updated CSV to S3
s3.upload_file(local_path, s3_bucket, key)

print(f"Uploaded to s3://{s3_bucket}/{key}")

Test loss: 0.6034657321636032 Test acc: 0.7028301886792453
Uploaded to s3://sagemaker-us-east-1-549206572067/cat-landmarks-project/benchmarks/cnn_training_metrics_v1.csv


In [111]:
# Save the trained model currently in memory after 5 epochs

import torch
import os
model_dir = "/tmp"
model_path = os.path.join(model_dir, "model_cls.pth")

#model_path = "/home/sagemaker-user/model_cls.pth"
torch.save(model.state_dict(), model_path)

print("Saved trained model to:", model_path)


Saved trained model to: /tmp/model_cls.pth


### Benchmark comparision to CNN

In [107]:
brightness = 0.40   # from brightness_baseline_v1.csv
cnn_val = metrics_df["val_acc"].max()

print("CNN Best Val Acc:", cnn_val)
print("Improvement:", cnn_val - brightness)


CNN Best Val Acc: 0.7005940594059406
Improvement: 0.30059405940594053


## Endpoint creation/ deployment

In [155]:
# Move model + create tar.gz for SageMaker

import os

model_dir = "/tmp"
model_path = os.path.join(model_dir, "model_cls.pth")


!rm -rf model model_fixed.tar.gz
!mkdir -p model
!cp {model_path} model/
!cp inference.py model/
!tar -czvf model_fixed.tar.gz model




model/
model/model_cls.pth
model/inference.py


In [156]:
#  Upload packaged model to S3

import boto3

s3 = boto3.client("s3")

s3_bucket = "sagemaker-us-east-1-549206572067"   
endpoint_key = "cat-landmarks-project/endpoint/model_fixed.tar.gz"

s3.upload_file("model_fixed.tar.gz", s3_bucket, endpoint_key)

print(f"Uploaded to s3://{s3_bucket}/{endpoint_key}")

Uploaded to s3://sagemaker-us-east-1-549206572067/cat-landmarks-project/endpoint/model_fixed.tar.gz


In [158]:
# Update an existing endpoint to use the NEW model artifact without increasing instance count
# Creates: new Model -> new EndpointConfig -> UpdateEndpoint

import boto3
import time

region = "us-east-1"
sm = boto3.client("sagemaker", region_name=region)

endpoint_name = "pytorch-inference-2026-02-14-00-36-22-952" 
role_arn = sagemaker.get_execution_role()

model_name = f"catcls-fixed-model-{int(time.time())}"
endpoint_config_name = f"catcls-fixed-epc-{int(time.time())}"

model_data_url = "s3://sagemaker-us-east-1-549206572067/cat-landmarks-project/endpoint/model_fixed.tar.gz"
image_uri = sagemaker.image_uris.retrieve(
    framework="pytorch",
    region=region,
    version="1.13",
    py_version="py39",
    instance_type="ml.m5.xlarge",
    image_scope="inference",
)

# 1) Create SageMaker Model that points to the fixed artifact
sm.create_model(
    ModelName=model_name,
    ExecutionRoleArn=role_arn,
    PrimaryContainer={
        "Image": image_uri,
        "ModelDataUrl": model_data_url,
        "Environment": {
            # optional while debugging
            "SAGEMAKER_MODEL_SERVER_WORKERS": "1"
        },
    },
)

# 2) Create new EndpointConfig (same instance type/count as existing)
sm.create_endpoint_config(
    EndpointConfigName=endpoint_config_name,
    ProductionVariants=[{
        "VariantName": "AllTraffic",
        "ModelName": model_name,
        "InitialInstanceCount": 1,
        "InstanceType": "ml.m5.xlarge",
    }],
)

# 3) Update the existing endpoint in-place (no extra quota)
sm.update_endpoint(
    EndpointName=endpoint_name,
    EndpointConfigName=endpoint_config_name,
)

print("UpdateEndpoint started:", endpoint_name)


UpdateEndpoint started: pytorch-inference-2026-02-14-00-36-22-952


In [160]:
#  Verifying endpoint 

import boto3
import time

sm = boto3.client("sagemaker", region_name="us-east-1")
endpoint_name = "pytorch-inference-2026-02-14-00-36-22-952"

while True:
    desc = sm.describe_endpoint(EndpointName=endpoint_name)
    status = desc["EndpointStatus"]
    print("Status:", status)
    if status in ("InService", "Failed"):
        print("Reason:", desc.get("FailureReason", ""))
        break
    time.sleep(20)


Status: Updating
Status: Updating
Status: Updating
Status: InService
Reason: 


## Inference

In [161]:
# Reading a sample the test image bytes from S3
# Use IdentitySerializer so SageMaker sends raw bytes to input_fn
# Expect JSON back from output_fn

import boto3
from sagemaker.serializers import IdentitySerializer
from sagemaker.deserializers import JSONDeserializer

s3 = boto3.client("s3")

key = "cat-landmarks-project/raw/cats/images/CAT_04/00000900_022.jpg"
obj = s3.get_object(Bucket=s3_bucket, Key=key)
payload = obj["Body"].read()

predictor.serializer = IdentitySerializer(content_type="application/x-image")
predictor.deserializer = JSONDeserializer()

result = predictor.predict(payload)
print(result)


{'prediction': 1}


In [165]:

import boto3
import pandas as pd
from sagemaker.serializers import IdentitySerializer
from sagemaker.deserializers import JSONDeserializer

s3 = boto3.client("s3")

# Ensure sure predictor is already created and points to deployed endpoint
predictor.serializer = IdentitySerializer(content_type="application/x-image")
predictor.deserializer = JSONDeserializer()

def split_s3_uri(s3_uri: str):
    # s3://bucket/key...
    parts = s3_uri.replace("s3://", "").split("/", 1)
    return parts[0], parts[1]

results = []
sample_df = prod_df.head(50) 

for i, row in sample_df.iterrows():
    s3_uri = row["s3_uri"]
    bucket, key = split_s3_uri(s3_uri)

    payload = s3.get_object(Bucket=bucket, Key=key)["Body"].read()

    pred = predictor.predict(payload)  
    pred_label = int(pred["prediction"])

    
    results.append({
        "row_id": int(i),
        "s3_uri": s3_uri,
        "y_true": int(row["label"]) if "label" in row and pd.notna(row["label"]) else None,
        "y_pred": pred_label,
    })

pred_df = pd.DataFrame(results)
pred_df.head(20)

,row_id,s3_uri,y_true,y_pred
0,0,s3://sagemaker-us-east-1-549206572067/cat-land...,1,1
1,1,s3://sagemaker-us-east-1-549206572067/cat-land...,1,1
2,2,s3://sagemaker-us-east-1-549206572067/cat-land...,1,1
3,3,s3://sagemaker-us-east-1-549206572067/cat-land...,1,1
4,4,s3://sagemaker-us-east-1-549206572067/cat-land...,1,1
5,5,s3://sagemaker-us-east-1-549206572067/cat-land...,1,1
6,6,s3://sagemaker-us-east-1-549206572067/cat-land...,1,1
7,7,s3://sagemaker-us-east-1-549206572067/cat-land...,1,1
8,8,s3://sagemaker-us-east-1-549206572067/cat-land...,1,1
9,9,s3://sagemaker-us-east-1-549206572067/cat-land...,1,1


## Modelling Landmarks